# LI-6400 + CO2Dot — on-command dual acquisition to JSON

Every `INTERVAL` seconds the notebook **triggers** one full LI-6400 record and one CO2Dot measurement, then appends the pair to a JSON file. The LI-6400 record is forced on command with a single LPL word — **no AutoLog needed**.
- **LI-6400** (RS-232 → COM4, 57600 8N1): sending **`LPLog`** (Execute-as-LPL) forces a full tab-delimited log record — all ~40 LogList variables. `CO2R, CO2S, H2OR, H2OS, Photo, Cond, Ci, Trmmol, Press, Obs, HHMMSS` are named; every field is also kept in `raw`.
- **CO2Dot** (USB) via `co2dot.py`: BME688 `env` (T/P/RH/Gas) and spectrometer `spec_flash` (dark/lit/diff).

**One-time setup on the LI-6400 console:**
1. Comm Port config: **Incoming Comm = Execute as LPL** (then return to OPEN's Main Screen so it takes effect). LTerm can be Off.
2. **New Measurements** mode, measuring.
3. Open logging **to the Comm port** once: `Log → Escape → "Log to COMM port? Y"`. That's the destination `LPLog` writes to. **AutoLog is NOT needed** — the notebook triggers each record.

JSON is rewritten atomically after every sample; interrupt the run cell (■) to stop.

In [ ]:
import sys, os, json, time, datetime, re
from pathlib import Path
import serial

# import the CO2Dot client (co2dot.py lives in Scripts/Li-Control/, one level up)
_cands = [Path.cwd(), Path.cwd().parent, Path.cwd() / "Scripts" / "Li-Control",
          Path(r"c:/Users/LudovicoCaracciolo/Documents/Git-repo/CO2dot/Scripts/Li-Control")]
for _c in _cands:
    if (_c / "co2dot.py").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break
else:
    raise FileNotFoundError("co2dot.py not found near " + str(Path.cwd()))
from co2dot import CO2Dot

print("pyserial", serial.__version__, "| CO2Dot client imported")

## Configure

In [ ]:
PORT_LI         = "COM4"      # LI-6400 RS-232 bridge (check Device Manager)
BAUD_LI         = 57600        # confirmed from the console: 57600 8 1 N
INTERVAL        = 10.0         # seconds between triggered samples (notebook-driven)
NUM_SAMPLES     = None         # None = run until you interrupt; or set an integer count
OUTPUT_PATH     = "li6400_co2dot_log.json"
LED_CURRENT     = 1            # CO2Dot spec_flash LED current
INCLUDE_SPECTRA = True         # include CO2Dot dark/lit/diff spectra in each sample
LI_TRIGGER      = "LPLog"      # LPL word that forces one full LI-6400 record on command

## LI-6400 serial handle + log-record parser (on-command trigger)
`read_li6400_logrec` sends `LPLog` (Execute-as-LPL) to force one full record, then reads and parses the tab-delimited line. Every field goes into `raw`; the ones verified against the live display are named.

In [ ]:
# --- LI-6400 serial handle + LOG-RECORD parser ------------------------------
# Uses the LI-6400 log stream (LTerm Off, "Echo logged data to Comm: Yes"), which
# emits one complete tab-delimited record per logged observation, CRLF-terminated.
# This is reliable and complete (unlike the differential LTerm display mirror).
class Li6400:
    def __init__(self, port, baud=57600, *, data_bits=8, parity="N", stop_bits=1,
                 xonxoff=False, rtscts=False, timeout=0.3):
        bits = {7: serial.SEVENBITS, 8: serial.EIGHTBITS}[data_bits]
        par = {"N": serial.PARITY_NONE, "E": serial.PARITY_EVEN, "O": serial.PARITY_ODD}[parity]
        stop = {1: serial.STOPBITS_ONE, 2: serial.STOPBITS_TWO}[stop_bits]
        self.ser = serial.Serial(port, baud, bytesize=bits, parity=par, stopbits=stop,
                                 xonxoff=xonxoff, rtscts=rtscts, timeout=timeout, write_timeout=2.0)
        self.port, self.baud = port, baud

    def read_available(self, seconds=1.0):
        end = time.monotonic() + seconds
        buf = bytearray()
        while time.monotonic() < end:
            chunk = self.ser.read(self.ser.in_waiting or 1)
            if chunk:
                buf.extend(chunk)
        return bytes(buf)

    def send(self, line, eol="\n"):
        self.ser.write((line + eol).encode())

    def close(self):
        try:
            self.ser.close()
        except Exception:
            pass


# Column names verified by value against the live display; every field is also kept
# in raw[] so nothing is lost. (Confirm/extend the map with a header capture.)
LI_COLS = {0: "Obs", 1: "HHMMSS", 2: "FTime",
           4: "Photo", 5: "Cond", 6: "Ci", 7: "Trmmol",
           17: "CO2R", 18: "CO2S", 19: "H2OR", 20: "H2OS", 26: "Press"}


def parse_log_record(rec):
    """One tab-delimited LI-6400 log line -> {named cols..., n_fields, raw[list]}."""
    fields = rec.split("\t")
    out = {"n_fields": len(fields), "raw": []}
    for i, f in enumerate(fields):
        f = f.strip().strip('"')
        try:
            v = float(f)
        except ValueError:
            v = f                       # e.g. the HH:MM:SS time string
        out["raw"].append(v)
        if i in LI_COLS:
            out[LI_COLS[i]] = v
    return out


def read_li6400_logrec(li, max_s=5.0, chunk_s=0.5, min_tabs=30, trigger="LPLog"):
    """Trigger a full LI-6400 record ON COMMAND and parse it.

    Sends the LPL word `trigger` (default "LPLog"): with 'Incoming Comm: Execute as
    LPL' enabled, that forces a data record to the open Comm log destination — the
    programmatic equivalent of pressing Log (LI-COR's documented remote method). Then
    it reads the tab-delimited record the instrument emits. An echoed command line has
    no tabs and is ignored by the min_tabs filter. A record is 'complete' when a
    newline follows it and it has >= min_tabs tabs, so a partial line is never parsed.

    Returns {} if nothing arrives within max_s — i.e. no log is open to the Comm port,
    or the console isn't in New Measurements mode. Pass trigger=None to instead wait
    passively (e.g. if you drive logging with AutoLog).
    """
    if trigger:
        li.ser.reset_input_buffer()          # drop any stale/partial bytes first
        li.send(trigger)                     # "LPLog\n" -> LF (dec 10) fires Execute-as-LPL
    buf = ""
    end = time.monotonic() + max_s
    while time.monotonic() < end:
        buf += li.read_available(chunk_s).decode("latin-1")
        parts = buf.split("\n")
        recs = [p.replace("\r", "") for p in parts[:-1] if p.count("\t") >= min_tabs]
        if recs:
            return parse_log_record(recs[-1])
    return {}

## Connect both devices
Open the LI-6400 **first** so it owns COM4; `CO2Dot.connect()` then autodetects the CO2Dot on its own USB port (it skips COM4 because it's busy).

In [ ]:
li = Li6400(PORT_LI, BAUD_LI)
print(f"LI-6400 open on {li.port} @ {li.baud} baud")

dot = CO2Dot.connect()
print(f"CO2Dot connected: {dot.device} on {dot.port}")

## Check the LPL trigger link
Confirms Execute-as-LPL is live and the link is bidirectional: an LPL `comm print` should echo the text straight back out the serial port.

In [ ]:
li.ser.reset_input_buffer()
li.send('"LPL-OK" comm print')          # LPL: print this token back out the comm port
_resp = li.read_available(2.0).decode("latin-1", "replace")
print("round-trip:", repr(_resp))
print("OK" if "LPL-OK" in _resp else "no echo — is Incoming Comm = Execute as LPL, back at Main Screen?")

## One sample (preview)
`read_once()` triggers a full LI-6400 record (sends `LPLog`) and reads the CO2Dot. Run it 2–3× — the `Obs` number should **increment each time**, proving the trigger forces a fresh record. Empty `li6400` ⇒ no log open to the Comm port (or not New Measurements / Execute-as-LPL off).

In [ ]:
def acquire(li, dot):
    """One on-command sample: trigger a full LI-6400 record (LPLog) + read the CO2Dot."""
    li_vals = read_li6400_logrec(li, trigger=LI_TRIGGER)     # sends LPLog, reads the record
    ts = datetime.datetime.now().isoformat(timespec="milliseconds")
    dot_rec = {"device": dot.device}
    try:
        dot_rec["env"] = dot.env()
    except Exception as e:
        dot_rec["env_error"] = repr(e)
    if INCLUDE_SPECTRA:
        try:
            dot_rec["spec"] = dot.spec_flash(LED_CURRENT)
        except Exception as e:
            dot_rec["spec_error"] = repr(e)
    return {"timestamp": ts, "li6400": li_vals, "co2dot": dot_rec}


def read_once():
    """Trigger and return one full sample (both instruments), for manual/on-demand use."""
    return acquire(li, dot)


_s = read_once()
if not _s["li6400"]:
    print("LI-6400: (no record) — open a log to the Comm port (Log -> Escape -> COMM -> Y),")
    print("         be in New Measurements mode, and Incoming Comm = Execute as LPL")
else:
    print("LI-6400   :", {k: _s["li6400"].get(k) for k in ("Obs", "CO2R", "CO2S", "H2OR", "H2OS")})
print("CO2Dot env:", _s["co2dot"].get("env"))

## Run the timed logging loop → JSON
Triggers a full LI-6400 record + a CO2Dot measurement every `INTERVAL` seconds and appends to `OUTPUT_PATH`. **Interrupt the cell (■) to stop** — nothing is lost.

*Set `INCLUDE_SPECTRA = False` for smaller files, or ask for a JSON-Lines version for very long runs.*

In [ ]:
samples = []
_next = time.monotonic()
print(f"Triggering LI-6400 (LPLog) + CO2Dot every {INTERVAL:g}s -> {OUTPUT_PATH}")
print("Interrupt the cell to stop; JSON saved after each sample.\n")
try:
    while NUM_SAMPLES is None or len(samples) < NUM_SAMPLES:
        s = acquire(li, dot)
        if not s["li6400"]:
            print("  (no LI-6400 record — is a log open to the Comm port, New Measurements mode?)")
        samples.append(s)
        _tmp = OUTPUT_PATH + ".tmp"
        with open(_tmp, "w", encoding="utf-8") as f:
            json.dump(samples, f, indent=2)
        os.replace(_tmp, OUTPUT_PATH)
        li6, env = s["li6400"], s["co2dot"].get("env", {})
        print(f"[{s['timestamp']}] #{len(samples)}  Obs={li6.get('Obs')}  "
              f"CO2R={li6.get('CO2R')} CO2S={li6.get('CO2S')}  |  "
              f"dot T={env.get('T')} RH={env.get('RH')} Gas={env.get('Gas')}")
        _next += INTERVAL
        time.sleep(max(0.0, _next - time.monotonic()))
except KeyboardInterrupt:
    print(f"\nStopped. {len(samples)} samples saved to {OUTPUT_PATH}")

## Close both devices

In [ ]:
li.close()
dot.close()
print("closed both devices")